# M2 — Feature engineering (weekly SKU demand)

FORESIGHT forecasts at **weekly** granularity. Features use only **past** observations (lags and rolling stats) to avoid leakage.

Seasonal-naive baseline uses **lag_52** (same week last year).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.forecast import build_weekly_sales, engineer_features, FEATURE_COLS

sales = pd.read_csv(ROOT / "data/processed/sales_daily.csv")
sales["Date"] = pd.to_datetime(sales["Date"])

weekly = build_weekly_sales(sales)
print("Weekly rows (pre-cutoff):", weekly.shape)
print(weekly.head())

Weekly rows (pre-cutoff): (5200, 3)
      SKU       Date  Units_Sold
0  SKU001 2024-01-07         106
1  SKU001 2024-01-14          85
2  SKU001 2024-01-21         117
3  SKU001 2024-01-28          89
4  SKU001 2024-02-04         119


## Lag and rolling features

In [2]:
weekly_full, model_ready = engineer_features(weekly)
print("Full weekly with features:", weekly_full.shape)
print("Model-ready (no NaN features):", model_ready.shape)
print("Feature columns:", FEATURE_COLS)
model_ready[FEATURE_COLS + ["Units_Sold"]].describe().round(2)

Full weekly with features: (5200, 9)
Model-ready (no NaN features): (2600, 9)
Feature columns: ['lag_1', 'lag_4', 'rolling_mean_4', 'rolling_std_4', 'rolling_max_4', 'lag_52']


,lag_1,lag_4,rolling_mean_4,rolling_std_4,rolling_max_4,lag_52,Units_Sold
count,2600.00,2600.00,2600.00,2600.00,2600.00,2600.00,2600.00
mean,98.01,98.00,98.01,9.10,108.35,98.20,97.98
std,52.47,52.48,51.71,4.82,54.98,52.58,52.47
min,9.00,9.00,13.00,0.50,16.00,5.00,9.00
25%,54.00,54.00,55.00,5.44,64.00,54.00,54.00
50%,95.00,94.00,94.00,8.37,105.00,95.00,95.00
75%,135.25,135.25,136.00,11.85,148.00,136.00,136.00
max,253.00,253.00,227.75,31.14,253.00,242.00,253.00


## Sample SKU feature row (sanity check)

In [3]:
sample = model_ready[model_ready["SKU"] == "SKU001"].tail(3)
sample[["SKU", "Date", "Units_Sold"] + FEATURE_COLS]

,SKU,Date,Units_Sold,lag_1,lag_4,rolling_mean_4,rolling_std_4,rolling_max_4,lag_52
101,SKU001,2025-12-14,89,88.0,87.0,85.75,2.061553,88.0,94.0
102,SKU001,2025-12-21,91,89.0,84.0,86.25,2.629956,89.0,77.0
103,SKU001,2025-12-28,76,91.0,84.0,88.00,2.943920,91.0,105.0


Next: **03_model_preparation.ipynb** (train/test split) and **04_rolling_backtesting.ipynb** (WAPE vs seasonal naive).